# Retrieval Metrics
> Measure retrieval quality with precision, recall, MRR, nDCG, and MAP.

`RetrievalMetricsCalculator` evaluates how well your retrieval
pipeline surfaces relevant documents. It compares a ranked list of
retrieved items against a ground-truth set of relevant IDs.

**Time:** ~5 minutes

## Setup

In [ ]:
from anchor.evaluation import RetrievalMetricsCalculator
from anchor.models import ContextItem, SourceType

## Build Mock Retrieved Documents
We create five `ContextItem` instances that simulate retrieval
results ranked by decreasing score. Only some of them are
actually relevant.

In [ ]:
retrieved = [
    ContextItem(
        id=f"doc-{i}",
        content=f"Content about topic {i}",
        source=SourceType.RETRIEVAL,
        score=1.0 - i * 0.1,
        priority=5,
        token_count=10,
    )
    for i in range(5)
]

for item in retrieved:
    print(f"{item.id}  score={item.score:.1f}  tokens={item.token_count}")

## Define Ground Truth
The relevant set tells the calculator which documents *should*
have been retrieved.

In [ ]:
relevant_ids = ["doc-0", "doc-1", "doc-3"]

print(f"Total retrieved:  {len(retrieved)}")
print(f"Total relevant:   {len(relevant_ids)}")
print(f"Relevant IDs:     {relevant_ids}")

## Compute Metrics at k=5
`calculate()` returns an object with precision, recall, MRR,
nDCG, and MAP -- the standard information-retrieval metrics.

In [ ]:
calculator = RetrievalMetricsCalculator()

metrics = calculator.evaluate(
    retrieved=retrieved,
    relevant=relevant_ids,
    k=5,
)

print(f"Precision@5: {metrics.precision:.3f}")
print(f"Recall@5:    {metrics.recall:.3f}")
print(f"MRR:         {metrics.mrr:.3f}")
print(f"nDCG@5:      {metrics.ndcg:.3f}")
print(f"MAP:         {metrics.map:.3f}")

## Compute Metrics at k=3
Changing *k* controls how many top results are considered.

In [ ]:
metrics_at_3 = calculator.evaluate(
    retrieved=retrieved,
    relevant=relevant_ids,
    k=3,
)

print(f"Precision@3: {metrics_at_3.precision:.3f}")
print(f"Recall@3:    {metrics_at_3.recall:.3f}")
print(f"MRR:         {metrics_at_3.mrr:.3f}")
print(f"nDCG@3:      {metrics_at_3.ndcg:.3f}")
print(f"MAP:         {metrics_at_3.map:.3f}")

## Compare k Values Side by Side
A quick table shows how the cut-off affects each metric.

In [ ]:
print(f"{'Metric':<12} {'k=3':>8} {'k=5':>8}")
print("-" * 30)
for name in ['precision', 'recall', 'mrr', 'ndcg', 'map']:
    v3 = getattr(metrics_at_3, name)
    v5 = getattr(metrics, name)
    print(f"{name:<12} {v3:>8.3f} {v5:>8.3f}")

## Key Takeaways

- `RetrievalMetricsCalculator` computes standard IR metrics in one call.
- Precision measures how many retrieved docs are relevant; recall
  measures how many relevant docs were retrieved.
- MRR reflects where the *first* relevant document appears in the ranking.
- nDCG and MAP account for the position of *all* relevant documents.
- Adjusting *k* lets you evaluate top-N retrieval quality.